<a href="https://colab.research.google.com/github/sanadv/Classical_Machine_Learning/blob/main/Classical_Machine_Leaning_Chapter_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Derivative of the sigmoid function
def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

# Log-loss (binary cross-entropy) function
def log_loss(y_true, y_pred):
    return - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Derivative of the log-loss function
def log_loss_derivative(y_true, y_pred):
    return (y_pred - y_true) / (y_pred * (1 - y_pred))

# Define the neural network structure based on the diagram
class SimpleNeuralNetwork:
    def __init__(self):
        # Initialize weights as per the given diagram
        # Initialize weights and biases randomly
        self.weights_1 = np.random.randn(2, 3)  # Random weights to Neuron1 and Neuron2
        self.bias_1 = np.random.randn(2)        # Random biases for Neuron1 and Neuron2
        self.weights_2 = np.random.randn(2)     # Random weights to Neuron3
        self.bias_2 = np.random.randn()         # Random bias for Neuron3

    # Forward pass
    def forward(self, inputs):
        # Layer 1 (hidden layer)
        self.z1 = np.dot(self.weights_1, inputs) + self.bias_1
        self.a1 = sigmoid(self.z1)  # Activation of Neuron1 and Neuron2

        # Layer 2 (output layer)
        self.z2 = np.dot(self.weights_2, self.a1) + self.bias_2
        self.a2 = sigmoid(self.z2)  # Output of Neuron3
        return self.a2

    # Backward pass (gradient computation)
    def backward(self, inputs, y_true):
        # Calculate output error
        y_pred = self.a2
        output_error = log_loss_derivative(y_true, y_pred) * sigmoid_derivative(self.z2)

        # Calculate gradients for weights between Neuron3 and hidden layer
        grad_weights_2 = output_error * self.a1
        grad_bias_2 = output_error

        # Calculate hidden layer errors
        hidden_errors = output_error * self.weights_2 * sigmoid_derivative(self.z1)

        # Calculate gradients for weights between inputs and hidden layer
        grad_weights_1 = np.outer(hidden_errors, inputs)
        grad_bias_1 = hidden_errors

        return grad_weights_1, grad_bias_1, grad_weights_2, grad_bias_2

    # Training function
    def train(self, X, y, epochs=1000, learning_rate=0.01):
        for epoch in range(epochs):
            for i in range(len(X)):
                inputs = X[i]
                y_true = y[i]

                # Forward pass
                self.forward(inputs)

                # Backward pass
                grad_w1, grad_b1, grad_w2, grad_b2 = self.backward(inputs, y_true)

                # Update weights and biases
                self.weights_1 -= learning_rate * grad_w1
                self.bias_1 -= learning_rate * grad_b1
                self.weights_2 -= learning_rate * grad_w2
                self.bias_2 -= learning_rate * grad_b2

    # Prediction function
    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            pred = self.forward(X[i])
            predictions.append(1 if pred >= 0.5 else 0)
        return predictions

# Load the dataset
import pandas as pd

# Load the dataset from the provided input
data = pd.read_csv('diabetes_data.csv')

# Prepare the input features and target
X = data[['Age', 'BMI', 'BGL']].values
y = data['Class'].values

# Scale the input features (optional but improves performance)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Initialize and train the neural network
nn = SimpleNeuralNetwork()
nn.train(X, y, epochs=100, learning_rate=0.01)

# Make predictions
predictions = nn.predict(X)

# Evaluate the model
accuracy = np.mean(predictions == y)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Display the predictions
results = pd.DataFrame({
    'Age': data['Age'],
    'BMI': data['BMI'],
    'Blood Glucose Level': data['BGL'],
    'Actual': y,
    'Predicted': predictions
})
print(results)


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Step 1: Load the dataset from a CSV file
data = pd.read_csv('customer_data.csv')

# Step 2: Inspect the dataset (uncomment if you want to see the data)
# print(data.head())

# Step 3: Preprocess the dataset
# Extract features and target
features = ['Customer Age', 'Purchase Frequency', 'Total Spending Last Year', 'Product Category Preference']
X = data[features]

# Encode the target labels (High-Value Customer, Loyal Customer, etc.)
label_encoder = LabelEncoder()
data['Class'] = label_encoder.fit_transform(data['Class'])  # Encode class labels as integers
y = data['Class']

# Convert labels to one-hot encoding
y = to_categorical(y, num_classes=4)

# Step 4: Normalize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Step 5: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Build the model
model = Sequential([
    Dense(4, activation='relu', input_shape=(4,)),  # First hidden layer with 4 neurons
    Dense(3, activation='relu'),                    # Second hidden layer with 3 neurons
    Dense(4, activation='softmax')                  # Output layer with 4 neurons (Softmax)
])

# Step 7: Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Step 8: Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=8, validation_split=0.2)

# Step 9: Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

# Step 10: Make predictions (optional)
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)

# Convert predicted classes back to original labels
predicted_labels = label_encoder.inverse_transform(predicted_classes)

# Print the first 10 predictions
print("Sample Predictions:", predicted_labels[:10])
